# Notebook 03 — RAG Pipeline

**Prerequisite:** `01_setup_data_prep.ipynb` đã chạy xong.

**Embedding model:** `BAAI/bge-m3` via FlagEmbedding (dense vector, dim=1024)  
**Vector store:** FAISS `IndexFlatIP` (cosine similarity)  
**Strategy:** Parent-child retrieval — embed child chunks, inject parent text vào LLM prompt

**Steps:**
1. Load `all_chunks.jsonl` (child) + `parent_chunks.jsonl` (parent map)
2. Encode child chunks với BGE-M3
3. Build + save FAISS index
4. Định nghĩa `retrieve()` + `format_context()`
5. Reusable `load_retriever()` helper cho NB05/NB07
6. Sanity check Recall@5 (source-file level)
7. Sample retrieval demo

## 0. Setup

In [30]:
from pathlib import Path
import os

if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = Path('../')

print(f'Base: {BASE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base: /content/drive/MyDrive/NLP_Final/Source


In [19]:
# %%capture
!pip install -U faiss-cpu FlagEmbedding -q
print('Dependencies installed.')

Dependencies installed.


## 1. Load Chunks & Parent Map

In [20]:
import json

CHUNKS_PATH  = BASE / 'data/chunks/all_chunks.jsonl'
PARENTS_PATH = BASE / 'data/chunks/parent_chunks.jsonl'

# Child chunks — dùng để build embedding index
index_chunks: list[dict] = []
with open(CHUNKS_PATH, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            index_chunks.append(json.loads(line))

# Parent map: parent_chunk_id → full paragraph text
parent_map: dict[str, str] = {}
with open(PARENTS_PATH, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            parent_map[p['parent_chunk_id']] = p['text']

print(f'Child chunks  : {len(index_chunks)}')
print(f'Parent chunks : {len(parent_map)}')
print(f'Files covered : {len(set(c["source_file"] for c in index_chunks))}/20')
print(f'Schema        : {list(index_chunks[0].keys())}')

Child chunks  : 280
Parent chunks : 232
Files covered : 20/20
Schema        : ['chunk_id', 'parent_chunk_id', 'text', 'source_file', 'char_length']


## 2. Load BGE-M3 Embedding Model

`BAAI/bge-m3` — multilingual model, dense dim=1024, max 8192 tokens.   
Chạy FP16 trên Colab T4 (~2 GB VRAM).

In [21]:
from FlagEmbedding import BGEM3FlagModel
from dotenv import load_dotenv
load_dotenv(os.path.join(BASE, '.env'))
HF_TOKEN = os.environ.get('HF_TOKEN')

EMBED_MODEL = 'BAAI/bge-m3'
embedder = BGEM3FlagModel(EMBED_MODEL, use_fp16=True, token=HF_TOKEN)

print(f'Model: {EMBED_MODEL}')
print(f'FP16 : True  |  Max length: 8192 tokens')

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model: BAAI/bge-m3
FP16 : True  |  Max length: 8192 tokens


## 3. Encode Child Chunks

BGE-M3 trả về `dense_vecs` đã L2-normalized → dùng `IndexFlatIP` = cosine similarity.

In [22]:
import numpy as np

texts = [c['text'] for c in index_chunks]

print(f'Encoding {len(texts)} chunks with BGE-M3...')
output = embedder.encode(
    texts,
    batch_size=12,
    max_length=8192,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)
embeddings = np.array(output['dense_vecs'], dtype='float32')
print(f'Embeddings shape: {embeddings.shape}')  # (N, 1024)

Encoding 280 chunks with BGE-M3...


Inference Embeddings: 100%|██████████| 24/24 [00:05<00:00,  4.31it/s]

Embeddings shape: (280, 1024)


## 4. Build FAISS Index

In [23]:
import faiss

DIMENSION = embeddings.shape[1]  # 1024

# IndexFlatIP: exact inner product search
# BGE-M3 dense_vecs already L2-normalized → IP == cosine similarity
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)

print(f'FAISS index: {index.ntotal} vectors  |  dim={DIMENSION}')

FAISS index: 280 vectors  |  dim=1024


## 5. Save Index + Metadata

In [33]:
import pickle

INDEX_DIR  = BASE / 'data/vector_store/faiss_index'
INDEX_PATH = INDEX_DIR / 'index.faiss'
META_PATH  = INDEX_DIR / 'index.pkl'

INDEX_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(INDEX_PATH))

with open(META_PATH, 'wb') as f:
    pickle.dump(index_chunks, f)

print(f'Saved FAISS index    -> {INDEX_PATH}')
print(f'Saved chunk metadata -> {META_PATH}')

Saved FAISS index    -> /content/drive/MyDrive/NLP_Final/Source/data/vector_store/faiss_index/index.faiss
Saved chunk metadata -> /content/drive/MyDrive/NLP_Final/Source/data/vector_store/faiss_index/index.pkl


## 6. Retriever & format_context

In [34]:
def retrieve(query: str, k: int = 5) -> list[dict]:
    """Return top-k child chunks by BGE-M3 dense similarity."""
    out   = embedder.encode([query], return_dense=True, return_sparse=False, return_colbert_vecs=False)
    q_vec = np.array(out['dense_vecs'], dtype='float32')
    scores, indices = index.search(q_vec, k)
    return [
        {'chunk': index_chunks[i], 'score': float(scores[0][j])}
        for j, i in enumerate(indices[0])
        if i < len(index_chunks)
    ]


def format_context(results: list[dict], top_n: int = 3) -> str:
    """
    Parent-child retrieval strategy:
      1. Retrieve child chunks (accurate dense embedding)
      2. Look up full parent text via parent_map
      3. Dedup by parent_chunk_id
    """
    seen:  set[str]  = set()
    parts: list[str] = []
    for r in results:
        pid = r['chunk'].get('parent_chunk_id', '')
        if pid and pid in seen:
            continue
        if pid:
            seen.add(pid)
        ctx = parent_map.get(pid) or r['chunk']['text']
        parts.append(f'[{len(parts) + 1}] {ctx}')
        if len(parts) >= top_n:
            break
    return '\n\n'.join(parts)


print('retrieve() and format_context() ready.')

retrieve() and format_context() ready.


## 7. Reusable load_retriever() Helper

Dùng trong NB05 (experiments) và NB07 (Gradio demo).

In [35]:
def load_retriever(base):
    """
    Load toàn bộ RAG components từ disk.
    Returns: (retrieve_fn, format_context_fn)
    """
    import faiss, pickle, json, numpy as np
    from pathlib import Path
    from FlagEmbedding import BGEM3FlagModel

    base = Path(base)

    _index = faiss.read_index(str(base / 'data/vector_store/faiss_index/index.faiss'))
    with open(base / 'data/vector_store/faiss_index/index.pkl', 'rb') as f:
        _chunks = pickle.load(f)

    _parent_map: dict[str, str] = {}
    pp = base / 'data/chunks/parent_chunks_v2.jsonl'
    if pp.exists():
        with open(pp, encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    p = json.loads(line)
                    _parent_map[p['parent_chunk_id']] = p['text']

    _emb = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

    def _retrieve(query: str, k: int = 5) -> list[dict]:
        out   = _emb.encode([query], return_dense=True, return_sparse=False, return_colbert_vecs=False)
        q_vec = np.array(out['dense_vecs'], dtype='float32')
        scores, indices = _index.search(q_vec, k)
        return [
            {'chunk': _chunks[i], 'score': float(scores[0][j])}
            for j, i in enumerate(indices[0])
            if i < len(_chunks)
        ]

    def _format_context(results: list[dict], top_n: int = 3) -> str:
        seen, parts = set(), []
        for r in results:
            pid = r['chunk'].get('parent_chunk_id', '')
            if pid and pid in seen: continue
            if pid: seen.add(pid)
            ctx = _parent_map.get(pid) or r['chunk']['text']
            parts.append(f'[{len(parts) + 1}] {ctx}')
            if len(parts) >= top_n: break
        return '\n\n'.join(parts)

    print(f'Retriever: {_index.ntotal} vectors (BGE-M3, dim={_index.d}) | {len(_parent_map)} parents')
    return _retrieve, _format_context


print('load_retriever() helper defined.')

load_retriever() helper defined.


## 8. Sanity Check — Source Recall@5

> `qa_test.jsonl` dùng `parent_chunk_id` format v1 — recall đo ở **document level** (source_file).  
> Sau khi re-generate QA với `02_qa_generation.ipynb`, có thể đo parent-level recall chính xác hơn.

In [36]:
test_pairs: list[dict] = []
qa_path = BASE / 'data/qa_filtered/qa_test.jsonl'
if qa_path.exists():
    with open(qa_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                test_pairs.append(json.loads(line))

print(f'Test pairs: {len(test_pairs)}')

hits_at_5 = 0
sanity    = test_pairs[:10]

for pair in sanity:
    gold = pair.get('source_file', '')
    results = retrieve(pair['question'], k=5)
    retrieved_files = [r['chunk']['source_file'] for r in results]
    hit = gold in retrieved_files
    hits_at_5 += int(hit)
    tag = 'HIT ' if hit else 'MISS'
    print(f'[{tag}] {pair["question"][:70]}')
    if not hit:
        print(f'       gold: {gold}')
        print(f'       got : {set(retrieved_files)}')

recall = hits_at_5 / len(sanity) if sanity else 0
print(f'\nSource Recall@5 (10 samples): {hits_at_5}/10 = {recall:.0%}')
if recall < 0.8:
    print('Below 80% — check chunk quality or embedding model.')
else:
    print('Good retrieval quality.')

Test pairs: 284
[MISS] Em nghe nói không được ghi âm, chụp ảnh trong giờ học. Có đúng không ạ
       gold: (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
       got : {'25.2020-Noi quy phong thi.txt', 'QD 2793 - Nội quy thi trực tuyến 19.9.2023.txt', 'PHỤ LỤC NHỮNG ĐIỀU SINH VIÊN KHÔNG ĐƯỢC LÀM; NỘI DUNG VI PHẠM VÀ HÌNH THỨC XỬ LÝ SINH VIÊN.txt'}
[HIT ] Em thấy hướng dẫn có nói tránh vừa đi vừa dùng điện thoại. Có bắt buộc
[HIT ] Cho em hỏi, khi gặp bạn bè trong trường thì có cần chào không? Hay chỉ
[MISS] Sinh viên có được phép dùng tiền hoặc quà cáp để nhờ thầy cô giúp đỡ v
       gold: (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
       got : {'Quy chế Công tác sinh viên.txt', 'Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt'}
[HIT ] Nếu em muốn đăng một thông tin gì đó lên mạng xã hội, em cần làm những
[HIT ] Nếu em gặp vấn đề bất thường trên mạng liên quan đến trường, em nên là
[HIT ] Dạ, cho em hỏi hướng dẫn về giao tiếp ứng xử của trường mình có nh

## 9. Full Recall@5 on Entire Test Set

In [37]:
from tqdm import tqdm

if not test_pairs:
    print('No test pairs found.')
else:
    hits = 0
    for pair in tqdm(test_pairs, desc='Recall@5'):
        gold    = pair.get('source_file', '')
        results = retrieve(pair['question'], k=5)
        if gold in [r['chunk']['source_file'] for r in results]:
            hits += 1
    print(f'Source Recall@5 (full test set): {hits}/{len(test_pairs)} = {hits/len(test_pairs):.1%}')

Recall@5: 100%|██████████| 284/284 [00:13<00:00, 21.36it/s]

Source Recall@5 (full test set): 260/284 = 91.5%


## 10. Sample Retrieval Demo

In [38]:
DEMO_QUERIES = [
    'Dieu kien de duoc xet hoc bong khuyen khich hoc tap la gi?',
    'Sinh vien bi dinh chi hoc tap trong truong hop nao?',
    'Quy dinh ve trang phuc cua sinh vien trong truong nhu the nao?',
]

for query in DEMO_QUERIES:
    results = retrieve(query, k=5)
    print(f'Query: {query}')
    print('-' * 70)
    for i, r in enumerate(results):
        c = r['chunk']
        print(f'  [{i+1}] score={r["score"]:.4f}  src={c["source_file"]}')
        print(f'       {c["text"][:150]}...')
    print()

Query: Dieu kien de duoc xet hoc bong khuyen khich hoc tap la gi?
----------------------------------------------------------------------
  [1] score=0.3546  src=Bộ Quy tắc ứng xử của người học.txt
       BỘ QUY TẮC ỨNG XỬ CỦA NGƯỜI HỌC
I. MỤC TIÊU VÀ NGUYÊN TẮC:
1. Bộ Quy tắc ứng xử của người học (gọi tắt là “Quy tắc này”) được ban hành nhằm tạo lập mô...
  [2] score=0.3457  src=Quy dinh cong nhan ket qua hoc chi va chuyen doi tin chi.txt
       # TỔNG LIÊN ĐOÀN LAO ĐỘNG VIỆT NAM
# TRƯỜNG ĐẠI HỌC TÔN ĐỨC THẮNG
**CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM**
**Độc lập – Tự do – Hạnh phúc**
**Số: 1785/QĐ...
  [3] score=0.3456  src=QĐ số 22 vv ban hành quy định kiểm soát và xử lý hành vi đạo văn các sản phẩm học thuật.txt
       # TỔNG LIÊN ĐOÀN LAO ĐỘNG VIỆT NAM
# TRƯỜNG ĐẠI HỌC TÔN ĐỨC THẮNG
**CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM**
**Độc lập – Tự do – Hạnh phúc**
**Số: 22..../...
  [4] score=0.3434  src=Quy dinh cong nhan ket qua hoc chi va chuyen doi tin chi.txt
       *Căn cứ Quyết định số 889/Q

In [39]:
# Demo format_context (parent-child injection)
query   = 'Dieu kien de duoc xet hoc bong khuyen khich hoc tap la gi?'
results = retrieve(query, k=5)
ctx     = format_context(results, top_n=3)

print(f'Context for LLM (3 parent passages):\n')
print(ctx[:1200], '...' if len(ctx) > 1200 else '')

Context for LLM (3 parent passages):

[1] BỘ QUY TẮC ỨNG XỬ CỦA NGƯỜI HỌC
I. MỤC TIÊU VÀ NGUYÊN TẮC:
1. Bộ Quy tắc ứng xử của người học (gọi tắt là “Quy tắc này”) được ban hành nhằm tạo lập môi trường học tập, nghiên cứu bình đẳng, công bằng, văn minh và hiệu quả. Người học được tôn trọng, có quyền tự do ứng xử theo nguyên tắc tuyệt đối tuân thủ pháp luật, tôn trọng quyền và lợi ích hợp pháp của Nhà nước, Nhà trường và của các tổ chức, cá nhân khác.
2. Quy tắc này bao gồm những nguyên tắc cơ bản trong ứng xử của người học trong phạm vi Trường Đại học Tôn Đức Thắng (gọi tắt là “Trường” hoặc “Nhà trường”) và trong một số quan hệ xã hội khác mà Quy tắc này điều chỉnh.
3. Quy tắc này và Quy chế Công tác sinh viên, Quy chế quản lý người học trình độ thạc sĩ, tiến sĩ là cơ sở pháp lý để người học có thể thực hiện quyền cơ bản và chịu các trách nhiệm trong học tập, ứng xử tại Trường và trong các quan hệ xã hội.
4. Trong trường hợp người học có những hành vi ứng xử đáng khen, Nhà trường áp dụn

---
**Artifacts saved:**

| File | Nội dung |
|------|----------|
| `data/vector_store/faiss_index/index.faiss` | FAISS IndexFlatIP (~280 vectors, dim=1024) |
| `data/vector_store/faiss_index/index.pkl` | Child chunk metadata list |

**Embedding model:** `BAAI/bge-m3` (FP16, dense dim=1024)  
**Next step:** Run `04_sft_training.ipynb`.